# Power BI Scanner API — Dependency & Access Control Reporting

This notebook uses the **Power BI Admin Scanner API** to scan all workspaces in your tenant.
It produces two Delta tables:
- `View_Dependency_Table` — column-level lineage from source views to Power BI datasets
- `Access_Control_Table` — per-report user access rights

Run this notebook in a **Microsoft Fabric Lakehouse** notebook environment.

In [ ]:
import os
import time
import json
import msal
import requests
import pandas as pd
import re
from datetime import datetime

## 1. Configure Authentication

Set your Azure AD service principal credentials.
Replace the values below or inject them via environment variables / Fabric secrets.

In [ ]:
PBI_TENANT_NAME = "XXX"
PBI_ADMIN_API_CLIENT_ID = "XXX"
PBI_ADMIN_API_SECRET = "XXX"

PBI_AUTHORITY = f'https://login.microsoftonline.com/{PBI_TENANT_NAME}'
PBI_SCOPES = ['https://analysis.windows.net/powerbi/api/.default']

WORKSPACES_PER_CHUNK = 100 #optional if you have a lot of workspaces
SCAN_TIMEOUT = 30
MAX_SCAN_STATUS_POLL = 10

## 2. Get Access Token

In [ ]:
def get_access_token():
    clientapp = msal.ConfidentialClientApplication(
        PBI_ADMIN_API_CLIENT_ID,
        authority=PBI_AUTHORITY,
        client_credential=PBI_ADMIN_API_SECRET
    )
    response = clientapp.acquire_token_silent(PBI_SCOPES, account=None)
    if not response:
        response = clientapp.acquire_token_for_client(scopes=PBI_SCOPES)
    return response['access_token']

access_token = get_access_token()

## 3. Define API Functions

In [ ]:
def scan_workspaces(access_token):
    url = 'https://api.powerbi.com/v1.0/myorg/admin/workspaces/modified?excludePersonalWorkspaces=True'
    headers = {'Authorization': 'Bearer ' + access_token}
    response = requests.get(url, headers=headers)
    return response.json()

def get_workspace_info(access_token, workspaces):
    url = 'https://api.powerbi.com/v1.0/myorg/admin/workspaces/getInfo?datasetExpressions=True&datasetSchema=True&datasourceDetails=True&getArtifactUsers=True&lineage=True'
    headers = {'Authorization': 'Bearer ' + access_token}
    response = requests.post(url, headers=headers, json={'workspaces': workspaces})
    return response.headers['location']

def get_scan_status(access_token, url):
    headers = {'Authorization': 'Bearer ' + access_token}
    response = requests.get(url, headers=headers)
    return response.json(), response.headers.get('location')

def get_scan_result(access_token, url):
    headers = {'Authorization': 'Bearer ' + access_token}
    response = requests.get(url, headers=headers)
    return response.json()

## 4. Run Workspace Scan

Scans all modified workspaces in chunks of `WORKSPACES_PER_CHUNK`.

In [ ]:
workspaces = scan_workspaces(access_token)

scan_results = {'workspaces': []}

In [ ]:

ws_index = 0
while ws_index < len(workspaces):
    chunk = workspaces[ws_index:ws_index + WORKSPACES_PER_CHUNK]
    ws_ids = [ws['id'] for ws in chunk]
    ws_index += WORKSPACES_PER_CHUNK

    scan_url = get_workspace_info(access_token, ws_ids)

    for _ in range(MAX_SCAN_STATUS_POLL):
        time.sleep(SCAN_TIMEOUT)
        status, location = get_scan_status(access_token, scan_url)
        if status['status'] == 'Succeeded':
            break

    result = get_scan_result(access_token, location)
    scan_results['workspaces'].extend(result.get('workspaces', []))

In [ ]:
print(scan_results)

## 5. (Optional) Save Raw Scan Result

Save the raw JSON to the Lakehouse Files area for archiving or reuse.

In [ ]:
output_path = f"/lakehouse/default/Files/Scanner_API/pbi_scannerapi_output_{datetime.today().strftime('%Y%m%d')}.json"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as file:
    json.dump(scan_results, file, ensure_ascii=False, indent=2)

## 6. Load JSON & Flatten to Pandas

In [ ]:
with open(output_path, 'r', encoding='utf-8') as file:
    scan_data = json.load(file)

In [ ]:
print(scan_data)

## 7. Build Dependency Table

For each dataset, iterates tables and columns to capture:
- Source connection and view name
- Whether the column is referenced in any DAX measure
- Whether the column is kept after Power Query selection

In [ ]:

dependencies = []

for workspace in scan_data.get("workspaces", []):
    for dataset in workspace.get("datasets", []):
        dataset_name = dataset.get("name")
        dataset_id = dataset.get("id")
        tables = dataset.get("tables", [])

        measure_list = []
        for table in tables:
            for measure in table.get("measures", []):
                measure_name = measure.get("name", "")
                expr = measure.get("expression", "")
                measure_list.append((measure_name, expr))

        for table in tables:
            table_name = table.get("name")
            source_expr = ""
            source = "Unknown"
            selected_columns = None
            view_name = None

            if "source" in table and isinstance(table["source"], list) and table["source"]:
                source_expr = table["source"][0].get("expression", "")

                source_match = re.search(r'Source\s*=\s*([a-zA-Z0-9_]+)', source_expr)
                if source_match:
                    source = source_match.group(1)

                if "ADDCOLUMNS" in source_expr:
                    source = "Power BI (Calculated Table)"

                select_match = re.search(r'Table\.SelectColumns\([^\[]+\[\s*([^\]]+)\s*\]\)', source_expr)
                if select_match:
                    selected_columns = [col.strip().strip('"') for col in select_match.group(1).split(',')]

                match_name_kind = re.search(r'Source\s*{[^}]*Name\s*=\s*"([^"]+)"[^}]*Kind\s*=\s*"View"', source_expr)
                if match_name_kind:
                    view_name = f"api.{match_name_kind.group(1)}"

                match_schema_item = re.search(r'Schema\s*=\s*"([^"]+)"\s*,\s*Item\s*=\s*"([^"]+)"', source_expr)
                if match_schema_item:
                    schema = match_schema_item.group(1)
                    item = match_schema_item.group(2)
                    view_name = f"{schema}.{item}"

            for column in table.get("columns", []):
                column_name = column.get("name")

                used_in_measure = False
                used_in_measure_names = []

                for measure_name, expr in measure_list:
                    if (
                        re.search(rf'\b{re.escape(column_name)}\b', expr, re.IGNORECASE)
                        or f"[{column_name}]" in expr
                        or f"{table_name}[{column_name}]" in expr
                    ):
                        used_in_measure = True
                        used_in_measure_names.append(measure_name)

                explicitly_selected = (
                    selected_columns is None or column_name in selected_columns
                )

                source_view_column = f"{source}.{view_name}.{column_name}" if view_name else None

                dependencies.append({
                    "Dataset_Name": dataset_name,
                    "Dataset_ID": dataset_id,
                    "Source": source,
                    "Table": table_name,
                    "Column": column_name,
                    "Source_Expression": source_expr,
                    "View_Name": view_name,
                    "Source_View_Column": source_view_column,
                    "Used_in_Measure": used_in_measure,
                    "Used_in_Measure_Names": ", ".join(used_in_measure_names),
                    "Kept_in_Power_Query": explicitly_selected
                })

# Create Pandas and Spark DataFrames
df_dependencies = pd.DataFrame(dependencies)
df_dependencies.columns = [col.replace(" ", "_") for col in df_dependencies.columns]
df_spark = spark.createDataFrame(df_dependencies)

# Save as Delta Table
df_spark.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("View_Dependency_Table")

In [ ]:
df = spark.sql("SELECT * FROM View_Dependency_Table")
display(df)

## 8. Build Access Control Table

Collects per-report user access rights and flags reports that are distributed via an Org App.

In [ ]:
access_control = []

# Collect all report/dataset IDs that are included in apps
app_included_ids = set()
for workspace in scan_data.get("workspaces", []):
    for app in workspace.get("OrgApp", []):
        for relation in app.get("relations", []):
            app_included_ids.add(relation.get("dependentOnArtifactId"))

# Loop through each workspace and collect access rights
for workspace in scan_data.get("workspaces", []):
    workspace_name = workspace.get("name", "")
    
    # Loop through each report and collect user access
    for report in workspace.get("reports", []):
        report_name = report.get("name")
        report_id = report.get("id")
        dataset_id = report.get("datasetId")
        
        # Find the corresponding dataset name using the ID
        dataset_name = next((d["name"] for d in workspace.get("datasets", []) if d["id"] == dataset_id), None)
        
        included_in_app = report_id in app_included_ids
        
        users = report.get("users", [])
        for user in users:
            access_control.append({
                "Report_Name": report_name,
                "Dataset_Name": dataset_name,
                "User_Display_Name": user.get("displayName"),
                "Email": user.get("emailAddress"),
                "Access_Right": user.get("reportUserAccessRight"),
                "Principal_Type": user.get("principalType"),
                "Included_In_App": included_in_app
            })

# Convert to Pandas DataFrame
df_access_control = pd.DataFrame(access_control)

# Make column names Delta-compliant
df_access_control.columns = [col.replace(" ", "_") for col in df_access_control.columns]

# Convert to Spark DataFrame
df_spark_access = spark.createDataFrame(df_access_control)

# Save to Delta Table
df_spark_access.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("Access_Control_Table")

In [ ]:
df = spark.sql("SELECT * FROM Access_Control_Table")
display(df)